In [ ]:
# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import json
import os

In [ ]:
# Paths
TRAIN_CSV = 'data/train_input.csv'
TEST_CSV = 'data/test_input.csv'
MODEL_PATH = 'student_model.joblib'
OUTPUT_JSON = 'test_results.json'
print('Train:', TRAIN_CSV)
print('Test :', TEST_CSV)
print('Model will be saved to', MODEL_PATH)
print('Results JSON will be saved to', OUTPUT_JSON)

Train: data/train_input.csv
Test : data/test_input.csv
Model will be saved to student_model.joblib
Results JSON will be saved to test_results.json


In [ ]:
# Load data
df = pd.read_csv(TRAIN_CSV)
df_test = pd.read_csv(TEST_CSV)
print('Train shape:', df.shape)
print('Test shape :', df_test.shape)
df.head()

Train shape: (9000, 7)
Test shape : (1001, 6)


,ID,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,1,7,99,Yes,9,1,91
1,2,4,82,No,4,2,65
2,3,8,51,Yes,7,2,45
3,4,5,52,Yes,5,2,36
4,5,7,75,No,8,5,66


## Feature engineering (club features)
We create combined features to help the model capture interactions:
- `Study_Effort` = `Hours Studied` * (`Sample Question Papers Practiced` + 1)
- `Academic_Backup` = weighted sum of `Previous Scores` and `Sample Question Papers Practiced`
- `Extracurricular_bin` = binary encoding of `Extracurricular Activities`

In [ ]:
# Create combined features on train and test
def add_combined_features(df):
    df = df.copy()
    df['Extracurricular_bin'] = df['Extracurricular Activities'].map({'Yes':1, 'No':0})
    df['Study_Effort'] = df['Hours Studied'] * (df['Sample Question Papers Practiced'] + 1)
    # normalize Previous Scores to 0-1 then combine
    df['Prev_norm'] = (df['Previous Scores'] - df['Previous Scores'].min()) / (df['Previous Scores'].max() - df['Previous Scores'].min())
    df['Pract_norm'] = (df['Sample Question Papers Practiced'] - df['Sample Question Papers Practiced'].min()) / (df['Sample Question Papers Practiced'].max() - df['Sample Question Papers Practiced'].min())
    df['Academic_Backup'] = 0.7 * df['Prev_norm'] + 0.3 * df['Pract_norm']
    # drop intermediate cols if present later
    return df

df = add_combined_features(df)
df_test = add_combined_features(df_test)
df[['Hours Studied','Sample Question Papers Practiced','Study_Effort','Previous Scores','Academic_Backup','Extracurricular_bin']].head()

,Hours Studied,Sample Question Papers Practiced,Study_Effort,Previous Scores,Academic_Backup,Extracurricular_bin
0,7,1,14,99,0.733333,1
1,4,2,12,82,0.564972,0
2,8,2,24,51,0.197175,1
3,5,2,15,52,0.209040,1
4,7,5,42,75,0.581921,0


In [5]:
# Prepare features and target
TARGET = 'Performance Index'
FEATURES = [
    'Hours Studied',
    'Previous Scores',
    'Sleep Hours',
    'Sample Question Papers Practiced',
    'Extracurricular_bin',
    'Study_Effort',
    'Academic_Backup'
]
# drop intermediate normalization columns before modelling
for col in ['Prev_norm','Pract_norm']:
    if col in df.columns: df.drop(columns=[col], inplace=True)
    if col in df_test.columns: df_test.drop(columns=[col], inplace=True)
X = df[FEATURES].copy()
y = df[TARGET].copy()
X_test = df_test[FEATURES].copy()

In [ ]:
# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train size:', X_train.shape, 'Val size:', X_val.shape)

In [6]:
# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train size:', X_train.shape, 'Val size:', X_val.shape)

# Build pipeline: scaler + RandomForestRegressor
num_features = FEATURES
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features)
], remainder='drop')
model = Pipeline([
    ('pre', preprocessor),
    ('rf', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
])
# Train
model.fit(X_train, y_train)
print('Model trained')

Train size: (7200, 7) Val size: (1800, 7)
Model trained


In [ ]:
# Evaluate on validation set (no leakage: model trained only on X_train)
y_pred_val = model.predict(X_val)
mse = mean_squared_error(y_val, y_pred_val)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_val, y_pred_val)
r2 = r2_score(y_val, y_pred_val)

# Also evaluate on training set to detect overfitting
y_pred_train = model.predict(X_train)
train_mse = mean_squared_error(y_train, y_pred_train)
train_rmse = np.sqrt(train_mse)
train_r2 = r2_score(y_train, y_pred_train)

print(f'Train  RMSE: {train_rmse:.4f}, R2: {train_r2:.4f}')
print(f'Val    RMSE: {rmse:.4f}, R2: {r2:.4f}')
print(f'Gap (Val-Train) RMSE: {rmse - train_rmse:.4f}')
print(f'Validation MSE : {mse:.4f}')
print(f'Validation RMSE: {rmse:.4f}')
print(f'Validation MAE : {mae:.4f}')
print(f'Validation R2  : {r2:.4f}')

metrics = {'mse': float(mse), 'rmse': float(rmse), 'mae': float(mae), 'r2': float(r2),
           'train_mse': float(train_mse), 'train_rmse': float(train_rmse), 'train_r2': float(train_r2),
           'cv_rmse_mean': float(cv_rmse)}
metrics

Validation MSE : 5.5552
Validation RMSE: 2.3570
Validation MAE : 1.8787
Validation R2  : 0.9852


{'mse': 5.55524799456595,
 'rmse': 2.3569573595137334,
 'mae': 1.8786811856661856,
 'r2': 0.9852008457204889}

In [8]:
# Save the trained pipeline (includes preprocessor and model)
joblib.dump(model, MODEL_PATH)
print('Saved model to', MODEL_PATH)

Saved model to student_model.joblib


In [9]:
# Predict on test set using the saved model and save results + metrics to JSON
loaded = joblib.load(MODEL_PATH)
X_test = df_test[FEATURES].copy()
preds = loaded.predict(X_test)
# prepare output structure
out = {
    'metrics': metrics,
    'predictions': []
}
for idx, p in zip(df_test['ID'], preds):
    out['predictions'].append({'ID': int(idx), 'Predicted Performance Index': float(p)})
# write JSON
with open(OUTPUT_JSON, 'w') as f:
    json.dump(out, f, indent=2)
print('Wrote predictions and metrics to', OUTPUT_JSON)

Wrote predictions and metrics to test_results.json
